# Practica 1 - 2C

## Juego con contrincante

Autor:
 - Adrián Redondo García

---

## Introducción

En esta practica vamos a modelizar un juego con contrincante usando el algoritmo Minimax.

Se trata de un juego tipo tres en raya pero con una peculiaridad, las piezas tienen ciertas características las cuales afectan a la hora de jugar.

---

## Componentes del juego

## Piezas

Las caracteristicas de las piezas son:
 - Altura: Alta y baja
 - Color:  Blanco o negro
 - Hueco:  Con y sin
 - Forma:  Circular, cuadrada y triangular

Para modelizar cada pieza en el juego y representar sus características, seguiremos el siguiente modelo:
 - altura -> 0: baja    1: alta
 - color  -> 0: blanco  1: negro
 - hueco  -> 0: no      1: si
 - forma  -> 0: circulo 1: cuadrado 2: triangulo

Estas características se guardarán en tuplas, representando una tupla una pieza. Por ejemplo:
```python
( 0, 1, 1, 0 ) # Pieza baja, negra, con hueco y circular
```

In [1]:
global estado

def generar_piezas(tam):
    
    if tam == 4:
        piezas = [ (0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 1, 0), (0, 0, 1, 1),
                  (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 1, 0), (0, 1, 1, 1),
                  (1, 0, 0, 0), (1, 0, 0, 1), (1, 0, 1, 0), (1, 0, 1, 1),
                  (1, 1, 0, 0), (1, 1, 0, 1), (1, 1, 1, 0), (1, 1, 1, 1) ]
    elif tam == 5:
        piezas = [ (0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 1, 0), (0, 0, 1, 1),
                  (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 1, 0), (0, 1, 1, 1),
                  (1, 0, 0, 0), (1, 0, 0, 1), (1, 0, 1, 0), (1, 0, 1, 1),
                  (1, 1, 0, 0), (1, 1, 0, 1), (1, 1, 1, 0), (1, 1, 1, 1),
                  (0, 0, 0, 2), (1, 0, 1, 2), (1, 1, 0, 2), (0, 1, 1, 2),
                  (0, 0, 0, 2), (0, 0, 0, 0), (0, 1, 1, 1), (1, 0, 0, 1),
                  (1, 1, 1, 0) ]
    elif tam == 6:
        piezas = [ (0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 1, 0), (0, 0, 1, 1),
                  (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 1, 0), (0, 1, 1, 1),
                  (1, 0, 0, 0), (1, 0, 0, 1), (1, 0, 1, 0), (1, 0, 1, 1),
                  (1, 1, 0, 0), (1, 1, 0, 1), (1, 1, 1, 0), (1, 1, 1, 1), 
                  (0, 0, 0, 2), (0, 0, 1, 2), (0, 1, 0, 2), (0, 1, 1, 2),
                  (1, 0, 0, 2), (1, 0, 1, 2), (1, 1, 0, 2), (1, 1, 1, 2),]

    return piezas

## Tablero

El tablero será una clase la cual tendra una matriz y varios métodos para la facilitar la aplicación del minimax.
Las coordenadas del tablero siguen la nomenclatura que se usa en el ajedrez, por ejemplo: d4.

Los pesos de cada tablero para la heurística estarán en una matriz aparte.

In [2]:
from copy import deepcopy

class Tablero:
    def __init__(self, matrix):
        self.set_matrix(matrix)

    def __eq__(self, other):
        return isinstance(other, Tablero) and self.matrix == other.matrix

    def set_matrix(self, matrix):
        self.matrix = deepcopy(matrix)

    def get_matrix(self):
        return deepcopy(self.matrix)

    def size(self):
        return len(self.matrix)

    # Devuelve una lista con todas las casillas vacías
    def empty_positions(self):
        n = self.size()
        libres = []

        for fil in range(n):
            for col in range(n):
                if self.matrix[fil][col] is None:
                    libres.append((fil, col))
        return libres

    # Coloca la pieza en el tablero
    def place_tile(self, row, col, tile):
        
        n = self.size()

        if not (0 <= row < n and 0 <= col < n):
            return False
        if self.matrix[row][col] is not None:
            return False

        self.matrix[row][col] = tile
        return True

    # Comprueba si el estado de la matriz es ganador
    # Vamos comprobando en direcciones vertical, horizontal y en las diagonales
    # principales
    def is_goal(self):

        tam = len(self.matrix)

        # Miramos cada atributo de la pieza
        for atrib in range(4):

            # ===== Vertical
            for col in range(tam):
                pri_pieza = self.matrix[0][col]
                if pri_pieza is None:
                    continue
                atrib_ref = pri_pieza[atrib]

                linea_ganadora = True
                for fil in range(1, tam):
                    pieza = self.matrix[fil][col]
                    if pieza is None or pieza[atrib] != atrib_ref:
                        linea_ganadora = False
                        break
                if linea_ganadora:
                    return True

            # Horizontal
            for fil in range(tam):
                pri_pieza = self.matrix[fil][0]
                if pri_pieza is None:
                    continue
                atrib_ref = pri_pieza[atrib]

                linea_ganadora = True
                for col in range(1, tam):
                    pieza = self.matrix[fil][col]
                    if pieza is None or pieza[atrib] != atrib_ref:
                        linea_ganadora = False
                        break
                if linea_ganadora:
                    return True

            # Diagonal principal
            pri_pieza = self.matrix[0][0]
            if pri_pieza is not None:
                atrib_ref = pri_pieza[atrib]
                linea_ganadora = True
                for i in range(1, tam):
                    pieza = self.matrix[i][i]
                    if pieza is None or pieza[atrib] != atrib_ref:
                        linea_ganadora = False
                        break
                if linea_ganadora:
                    return True

            # Diagonal secundaria
            pri_pieza = self.matrix[0][tam - 1]
            if pri_pieza is not None:
                atrib_ref = pri_pieza[atrib]
                linea_ganadora = True
                for i in range(1, tam):
                    pieza = self.matrix[i][tam - 1 - i]
                    if pieza is None or pieza[atrib] != atrib_ref:
                        linea_ganadora = False
                        break
                if linea_ganadora:
                    return True

        return False

    # 
    def utility(self):
        
        n = self.size()
        pesos = None
        total = 0

        pesos_tablero_4 = ((1, 1, 1, 1),
                           (1, 2, 2, 1),
                           (1, 2, 2, 1),
                           (1, 1, 1, 1))

        pesos_tablero_5 = ((1, 1, 1, 1, 1),
                           (1, 2, 2, 2, 1),
                           (1, 2, 3, 2, 1),
                           (1, 2, 2, 2, 1),
                           (1, 1, 1, 1, 1))

        pesos_tablero_6 = ((1, 1, 1, 1, 1, 1),
                           (1, 2, 2, 2, 2, 1),
                           (1, 2, 3, 3, 2, 1),
                           (1, 2, 3, 3, 2, 1),
                           (1, 2, 2, 2, 2, 1),
                           (1, 1, 1, 1, 1, 1))

        if n == 4:
            pesos = pesos_tablero_4
        elif n == 5:
            pesos = pesos_tablero_5
        elif n == 6:
            pesos = pesos_tablero_6

        for fil in range(n):
            for col in range(n):
                total = pesos[fil][col]
        
        return total


## Estado

El estado será representado mediante una tupla con las siguientes variables:
```python
estado = (tablero, bolsa, pieza_elegida, turno)
```

- tablero: el objeto Tablero
- bolsa: las piezas aun no colocadas
- pieza_elegida: la pieza seleccionada que debe usar el rival
- turno: de quien es el turno, si del humano o de la IA

## Inicialización

Con estas funciones generamos el estado inicial.
También hacen varias funciones.

In [3]:
import random
from typing import Tuple, List

def estado_inicial(tam):

    # Inicializamos el tablero, la bolsa con las piezas y el turno
    matriz = [[None for col in range(tam)] for row in range(tam)]
    tablero = Tablero(matriz)
    
    bolsa = generar_piezas(tam)
    turno = 0

    # Elegimos la primera pieza de la bolsa de forma aleatoria
    # y la eliminamos de esta
    pieza_elegida = random.choice(bolsa)
    bolsa = bolsa.copy()
    bolsa.remove(pieza_elegida)

    return (tablero, bolsa, pieza_elegida, turno)

In [4]:
def pieza_a_str(pieza):
    a, b, c, d = pieza
    return f"{a}{b}{c}{d}"

def convertir_coord(coord):
    coord = coord.strip().lower()
    col = ord(coord[0]) - 97
    fil = int(coord[1:]) - 1
    return fil, col

In [5]:
HUMANO = 0
IA = 1

# Realiza la accion de colocar la pieza en el tablero y alternar el turno

def aplicar_accion(estado, accion):
    tablero, bolsa, pieza_elegida, turno = estado
    fil, col, sig_pieza = accion

    # Hacemos copia del tablero
    tablero_nuevo = Tablero(tablero.get_matrix())

    # Colocar la pieza elegida
    tablero_nuevo.place_tile(fil, col, pieza_elegida)

    # Copiar bolsa y quitar sig_pieza
    nueva_bolsa = bolsa.copy()
    nueva_bolsa.remove(sig_pieza)  # quita una ocurrencia

    # Alternar turno
    nuevo_turno = HUMANO if turno == IA else IA

    if turno == HUMANO:
        nuevo_turno = IA
    else:
        nuevo_turno = HUMANO
    
    # La pieza elegida para el rival es la que acabas de elegir
    nueva_pieza_elegida = sig_pieza

    return (tablero_nuevo, nueva_bolsa, nueva_pieza_elegida, nuevo_turno)

In [6]:
# Funcion que se encarga del turno del jugador
def turno_manual(estado):

    clear_output(wait=True)

    tablero, bolsa, pieza_elegida, turno = estado
    n = tablero.size()

    libres = tablero.empty_positions()
    if not libres:
        return estado, True

    opciones = bolsa
    if not opciones:
        return estado, True

    # Escribir coordenada
    while True:
        coord = input("Introduce una coordenada (Ejemplo: a3): ")

        if len(coord) < 2:
            print("Formato inválido. Usa algo tipo a3.")
            continue

        try:
            fil, col = convertir_coord(coord)
        except:
            print("Formato inválido. Usa algo tipo a3.")
            continue

        if (fil, col) not in libres:
            print("Casilla no válida (ocupada o fuera).")
            continue

        break
    
    # Imprimimos las opciones
    for i, pieza in enumerate(opciones[:20]):
        print(f"  [{i}] {pieza_a_str(pieza)}")

    while True:
        try:
            id_pieza = int(input(f"Elige la pieza para el rival (0..{len(opciones) - 1}): "))
            if not (0 <= id_pieza < len(opciones)):
                print("Índice inválido.")
                continue
            sig_pieza = opciones[id_pieza]
            break
        except ValueError:
            print("Introduce un entero.")

    nuevo_estado = aplicar_accion(estado, (fil, col, sig_pieza))

    if nuevo_estado[0].is_goal():
        ganador_jugador = True
        return nuevo_estado, True

    return nuevo_estado, False

## Minimax

Algoritmo de Minimax con poda $\alpha-\beta$

In [7]:
import math

# Cuerpo principal del algoritmo
def miniMax(estado, nivel, profun_max, turno, alfa, beta, podar):

    tablero, bolsa, pieza_elegida, turno = estado

    # Si el tablero ya es goal, significa que el turno anterior acaba de ganar.
    # Como estamos "en" el turno de 'turno', si el goal ya está, el que mueve ahora ha perdido.
    if tablero.is_goal():
        
        M = 10**9
        
        if turno == IA:
            resul = -M
        else:
            resul = +M

        return (estado, resul, True)

    # Empate
    if len(tablero.empty_positions()) == 0 or len(bolsa) == 0:
        return (estado, 0, True)

    # Corte por profundidad
    if nivel >= profun_max:
        return (estado, tablero.utility(), True)

    # Sucesores
    sucesores = []
    for accion in acciones_legales(estado):
        sucesores.append(aplicar_accion(estado, accion))

    if len(sucesores) == 0:
        coste = tablero.utility()
        return (estado, coste, True)

    mejor_estado = None

    # MAX (IA)
    if turno == IA:
        valor_max = -math.inf
        for hijo in sucesores:
            _, utility, _ = miniMax(hijo, nivel + 1, profun_max, HUMANO, alfa, beta, podar)

            if utility > valor_max:
                valor_max = utility
                mejor_estado = hijo

            alfa = max(alfa, valor_max)
            if beta <= alfa:
                break

        return (mejor_estado, valor_max, False)

    # MIN (HUMANO)
    else:
        valor_min = math.inf
        for hijo in sucesores:
            _, utility, _ = miniMax(hijo, nivel + 1, profun_max, IA, alfa, beta, podar)

            if utility < valor_min:
                valor_min = utility
                mejor_estado = hijo

            beta = min(beta, valor_min)
            if beta <= alfa:
                break

        return (mejor_estado, valor_min, False)


In [8]:
# Realiza la accion de la IA
def performActionMinMax(estado, turno, profundidad):
    podar = False
    nivel = 0
    mejor_estado, mejor_valor, podar = miniMax(estado, nivel, profundidad, turno, -math.inf, math.inf, podar)
    return mejor_estado, mejor_valor


In [9]:
# Devuelve el mejor estado obtenido
def AIAction(estado, profundidad):
    mejor_estado, mejor_valor = performActionMinMax(estado, IA, profundidad)
    return mejor_estado


In [10]:
# Devuelve donde puede colocar la pieza la IA

def acciones_legales(estado):
    tablero, bolsa, pieza_elegida, turno = estado

    # Casillas libres
    posiciones = tablero.empty_positions()

    # Piezas posibles para entregar al rival
    sig_piezas = list(set(bolsa))

    # Todas las combinaciones válidas
    acciones = []
    for (fil, col) in posiciones:
        for pieza in sig_piezas:
            acciones.append((fil, col, pieza))

    return acciones


In [11]:
# Realiza el turno de la IA
def turno_ia_minimax(estado, profundidad):
    # calcula el mejor estado hijo con minimax
    mejor_estado, mejor_valor = performActionMinMax(estado, IA, profundidad)
    # chequeos de final de partida
    if mejor_estado[0].is_goal():
        return mejor_estado, True
    if len(mejor_estado[0].empty_positions()) == 0 or len(mejor_estado[1]) == 0:
        return mejor_estado, True
    return mejor_estado, False

## Representacion gráfica

Para la representación gráfica se usa una interfaz hecha en HTML.

Al elegir la siguiente pieza se imprimirán las fichas de las bolsa de la forma en la que han sido modelizadas por terminal.

In [12]:
from IPython.display import display
from ipywidgets import HTML

def get_content(fil, col, estado):
    
    tablero = estado[0].get_matrix()
    contenido = [None]               
    
    if tablero[fil][col] == (0,0,0,0):
        contenido[0] = "0000"
    elif tablero[fil][col] == (0,0,0,1):
        contenido[0] = "0001"
    elif tablero[fil][col] == (0,0,1,0):
        contenido[0] = "0010"
    elif tablero[fil][col] == (0,0,1,1):
        contenido[0] = "0011"
    elif tablero[fil][col] == (0,1,0,0):
        contenido[0] = "0100"
    elif tablero[fil][col] == (0,1,0,1):
        contenido[0] = "0101"
    elif tablero[fil][col] == (0,1,1,0):
        contenido[0] = "0110"
    elif tablero[fil][col] == (0,1,1,1):
        contenido[0] = "0111"
    elif tablero[fil][col] == (1,0,0,0):
        contenido[0] = "1000"
    elif tablero[fil][col] == (1,0,0,1):
        contenido[0] = "1001"
    elif tablero[fil][col] == (1,0,1,0):
        contenido[0] = "1010"
    elif tablero[fil][col] == (1,0,1,1):
        contenido[0] = "1011"
    elif tablero[fil][col] == (1,1,0,0):
        contenido[0] = "1100"
    elif tablero[fil][col] == (1,1,0,1):
        contenido[0] = "1101"
    elif tablero[fil][col] == (1,1,1,0):
        contenido[0] = "1110"
    elif tablero[fil][col] == (1,1,1,1):
        contenido[0] = "1111"
    elif tablero[fil][col] == (0,0,0,2):
        contenido[0] = "0002"
    elif tablero[fil][col] == (0,0,1,2):
        contenido[0] = "0012"
    elif tablero[fil][col] == (0,1,0,2):
        contenido[0] = "0102"
    elif tablero[fil][col] == (0,1,1,2):
        contenido[0] = "0112"
    elif tablero[fil][col] == (1,0,0,2):
        contenido[0] = "1002"
    elif tablero[fil][col] == (1,0,1,2):
        contenido[0] = "1012"
    elif tablero[fil][col] == (1,1,0,2):
        contenido[0] = "1102"
    elif tablero[fil][col] == (1,1,1,2):
        contenido[0] = "1112"
    elif tablero[fil][col] == None:
        contenido[0] = "vacio"

    return contenido

element_image = {
    "blanco": "./fichas/blanco.png",
    "0000": "./fichas/0000.png",
    "0001": "./fichas/0001.png",
    "0010": "./fichas/0010.png",
    "0011": "./fichas/0011.png",
    "0100": "./fichas/0100.png",
    "0101": "./fichas/0101.png",
    "0110": "./fichas/0110.png",
    "0111": "./fichas/0111.png",
    "1000": "./fichas/1000.png",
    "1001": "./fichas/1001.png",
    "1010": "./fichas/1010.png",
    "1011": "./fichas/1011.png",
    "1100": "./fichas/1100.png",
    "1101": "./fichas/1101.png",
    "1110": "./fichas/1110.png",
    "1111": "./fichas/1111.png",
    "0002": "./fichas/0002.png",
    "0012": "./fichas/0012.png",
    "0102": "./fichas/0102.png",
    "0112": "./fichas/0112.png",
    "1002": "./fichas/1002.png",
    "1012": "./fichas/1012.png",
    "1102": "./fichas/1102.png",
    "1112": "./fichas/1112.png",
    "vacio": "./fichas/vacio.png",
    "1": "./fichas/1.png",
    "2": "./fichas/2.png",
    "3": "./fichas/3.png",
    "4": "./fichas/4.png",
    "5": "./fichas/5.png",
    "6": "./fichas/6.png",
    "a": "./fichas/a.png",
    "b": "./fichas/b.png",
    "c": "./fichas/c.png",
    "d": "./fichas/d.png",
    "e": "./fichas/e.png",
    "f": "./fichas/f.png",
}

def get_html(estado):
    """ Muestra una representación gráfica del juego.

    Devuelve un "string" que contiene HTML

    """ 
    tablero, bolsa, pieza_elegida, turno = estado

    height = tablero.size()
    width = tablero.size()
    
    html_string = "<style> img.game {width: 60px !important; height: 60px !important;}</style><table>"

    new_row = "<tr>"
    end_row = "</tr>"
    
    cont_ind = 1

    for i in range(height):
        html_string+=new_row

        drawing = element_image[str(cont_ind)]
        html = '<td><img class="game" src="%s" alt=""></img></td>' % drawing
        html_string+=html
        cont_ind += 1

        for j in range(width):

            content = get_content(i,j,estado)
            drawing = element_image[content[0]]
            
            html = '<td><img class="game" src="%s" alt=""></img></td>' % drawing
            
            html_string+=html
        html_string+=end_row

    # Agregamos la columan final para las coordenadas de las columnas
    html_string += new_row

    # Hueco en blanco
    drawing = element_image["blanco"]
    html = '<td><img class="game" src="%s" alt=""></img></td>' % drawing
    html_string += html

    for i in range(width):
        letra = chr(97 + i)
        drawing = element_image[letra]
        html = '<td><img class="game" src="%s" alt=""></img></td>' % drawing
        html_string+=html

    html_string += "</table>"

    # Informacion

    var = pieza_a_str(pieza_elegida)
    drawing = element_image[var]
    
    html_string += "<br>"
    html_string += "<div style='display:flex; gap:20px; align-items:center;'>"

    html_string += f"<div><b>Piezas en bolsa:</b> {len(bolsa)}</div>"
    html_string += f"<div><b>Pieza elegida:</b><br><img class='game' src=\"{drawing}\"></div>"

    if ganador_IA:
        html_string += "<div><b>La IA ha ganado</div>"
    elif ganador_jugador:
        html_string += "<div><b>El jugador ha ganado</div>"
    elif empate:
        html_string += "<div><b>Hay un empate</div>"
    else:
        html_string += "</div>"

    return html_string

# Ejecución

Para poner en marcha el programa, primero hay que ejecutar todas las celdas de código anteriores. Tras eso tan solo hay que ejecutar la celda de abajo para dar comienzo a una partida.

La profundidad recomendada es 3, ya que a partir de 4 en adelanta tarda mucho en evaluar todas las posibles jugadas.

In [13]:
from IPython.display import display, clear_output
from ipywidgets import HTML

def main():

    # Variables para mostrar el resultado de la partida
    # en la interfaz
    global ganador_IA
    global ganador_jugador
    global empate
    
    ganador_IA = False
    ganador_jugador = False
    empate = False

    while True:
        try:
            tam = int(input("Elige tamaño de tablero (4, 5 o 6): "))
            if tam not in (4, 5, 6):
                continue
            break
        except ValueError:
            pass

    try:
        profundidad = int(input("Elige la profundidad del árbol para Minimax: "))
    except ValueError:
        profundidad = 2

    estado = estado_inicial(tam)

    fin = False

    while not fin:
        # Render del tablero HTML
        clear_output(wait=True)
        display(HTML(get_html(estado)))

        tablero, bolsa, pieza_elegida, turno = estado

        # Empate
        if len(tablero.empty_positions()) == 0 or len(bolsa) == 0:
            empate = True
            break

        # Turno HUMANO
        if turno == HUMANO:
            estado, fin = turno_manual(estado)

        # Turno IA (minimax)
        else:
            estado = AIAction(estado, profundidad)

            # Si el estado es el final, gana la IA
            if estado[0].is_goal():
                fin = True
                clear_output(wait=True)
                ganador_IA = True
                display(HTML(value=get_html(estado)))
            
            elif len(estado[0].empty_positions()) == 0 or len(estado[1]) == 0:
                fin = True
                clear_output(wait=True)
                empate = True
                display(HTML(get_html(estado)))
                
        t = estado[3]
        

main()

HTML(value='<style> img.game {width: 60px !important; height: 60px !important;}</style><table><tr><td><img cla…